# Dictionaries, Sets, and Reconciliation


```{contents}
:local:
:depth: 2
```


```{index} A Set Answers Membership; a Count Answers Multiplicity
```



```{index} A Set Answers Membership; a Count Answers Multiplicity
```

## A Set Answers Membership; a Count Answers Multiplicity



Two departments submit product IDs for reconciliation. A set represents which IDs occur, regardless of how many times they were entered. A dictionary of counts represents how often each ID occurs. Choosing between them changes the information retained, not merely the syntax of the container.

Let the warehouse IDs be {A, B, C} and the sales IDs be {B, C, D}. Intersection gives IDs recognized by both: {B, C}. Warehouse minus sales gives {A}; sales minus warehouse gives {D}. The direction of difference matters. A union gives all recognized IDs {A, B, C, D}; it does not establish that either department's quantities agree.

| Question | Operation | Result for this example |
| --- | --- | --- |
| Which IDs are shared? | Intersection | B, C |
| Which warehouse IDs are absent from sales? | Warehouse minus sales | A |
| Which sales IDs are absent from warehouse? | Sales minus warehouse | D |
| Which IDs appear anywhere? | Union | A, B, C, D |

Now suppose sales submits B twice. A set still contains one B. That is correct for unique membership and wrong for a task that must detect repeated submissions. A frequency dictionary can increment B's count to two. State whether duplicates are errors, repeated events, or information to discard before building either representation.

For related explanation, see the further reading below.[^development-reading]


```{index} Choose Equality Before Comparing the Groups
```



```{index} Choose Equality Before Comparing the Groups
```

## Choose Equality Before Comparing the Groups



If product IDs are case-insensitive under the business contract, B and b must use compatible equality in both collections. Normalizing only one side can produce false missing-ID reports. For hashed keys, equal keys must have equal hash codes under the selected policy. A good hash distribution improves performance but cannot repair inconsistent equality.

Do not mutate fields that determine a key's hash/equality while that key is stored. The entry may remain physically present in a bucket chosen by its old meaning while a new lookup examines another location. Immutable key data or remove/change/reinsert workflows make that boundary explicit.

Set-operation APIs also have mutation contracts. Some operations modify the receiving set; query-based forms may produce a sequence instead. If the original warehouse membership must remain available, work on a copy or choose a nonmutating approach deliberately. Assigning another variable to the same set object does not create independent storage.

Guided exercise: add B twice and lowercase a once to the sales input. Produce two answers: a unique-ID reconciliation under a stated case policy and a repeated-ID report. Explain which information a set has discarded and why the frequency report cannot be reconstructed from that set alone.


Keyed access and membership answer different questions. A dictionary maps an ID to a record. A set records whether an ID belongs to a group. Business reconciliation often needs set operations as well as individual lookups.

Use the same equality rule for all collections that participate in one calculation. The examples use ordinal case-insensitive equality for submitted symbolic IDs and preserve original sets when a report needs several answers.


```{index} Sets and Frequency Counts
```

## Sets and Frequency Counts




```{index} HashSet; set; frequency count
```

A set stores keys without associated record values. Use `HashSet<T>.Add` to detect a first occurrence, or a dictionary to count occurrences. Give both the same comparer when they must agree about what counts as equal.

In [1]:
using System;
using System.Collections.Generic;
using System.Linq;

string[] statuses = { "Open", "open", "Closed", "OPEN", "closed" };
var seen = new HashSet<string>(StringComparer.OrdinalIgnoreCase);
var counts = new Dictionary<string, int>(StringComparer.OrdinalIgnoreCase);
foreach (string status in statuses)
{
    seen.Add(status);
    counts.TryGetValue(status, out int count);
    counts[status] = count + 1;
}
Console.WriteLine($"distinct statuses={seen.Count}");
foreach (string status in new[] { "open", "closed" })
    Console.WriteLine($"{status}: {counts[status]}");

distinct statuses=2
open: 3
closed: 2


The set has two distinct statuses, and the counts are open three and closed two. The example prints keys in an explicit order; a hash collection's enumeration order is not a sorting contract. Sort keys separately when a report requires an order. Neither a set nor a dictionary supplies the priority-selection operation of a heap.

```{index} Union, Intersection, and Difference
```

## Union, Intersection, and Difference




```{index} Union, Intersection, and Difference
```


Let registered contain everyone registered for a workshop and attended contain everyone who checked in. Their intersection contains registered attendees. Registered minus attended identifies no-shows. Attended minus registered identifies walk-ins. Their union contains everyone appearing in either source.

HashSet operations such as IntersectWith and ExceptWith mutate the receiving set. Construct a copy first when you need to retain both source sets. Enumeration is not a sorted-report contract: copy to a list and sort explicitly for predictable output.


In [1]:
using System;
using System.Collections.Generic;

var registered=new HashSet<string>(new[]{"A","B","C"},StringComparer.OrdinalIgnoreCase);
var attended=new HashSet<string>(new[]{"b","C","D"},StringComparer.OrdinalIgnoreCase);
var both=new HashSet<string>(registered,StringComparer.OrdinalIgnoreCase);both.IntersectWith(attended);
var absent=new HashSet<string>(registered,StringComparer.OrdinalIgnoreCase);absent.ExceptWith(attended);
var all=new HashSet<string>(registered,StringComparer.OrdinalIgnoreCase);all.UnionWith(attended);
foreach(var group in new[]{both,absent,all}){var report=new List<string>(group);report.Sort(StringComparer.OrdinalIgnoreCase);Console.WriteLine(string.Join(",",report));}
Console.WriteLine($"Sources: {registered.Count}, {attended.Count}");


B,C
A
A,B,C,D
Sources: 3, 3


```{index} Lookup and Missing Values
```

## Lookup and Missing Values




```{index} Lookup and Missing Values
```


TryGetValue separates an absent key from a stored default value. Do not use a balance of zero, an empty owner name, or a null-like sentinel as an implicit membership test unless the contract explicitly excludes those values.

Choose Add when duplicate keys are errors; use indexer assignment when replacement is intended. If imports may repeat customer IDs, state whether a repeated row updates, accumulates, or rejects. Hashing provides access; it does not choose the business policy.


In [1]:
using System;
using System.Collections.Generic;

var units=new Dictionary<string,int>(StringComparer.OrdinalIgnoreCase);
units["P100"]=0;
foreach(string id in new[]{"p100","missing"})
    Console.WriteLine(units.TryGetValue(id,out int count)?$"{id}: stored {count}":$"{id}: missing");


p100: stored 0
missing: missing


```{index} Keys Must Keep Their Meaning
```

## Keys Must Keep Their Meaning




```{index} Keys Must Keep Their Meaning
```


Do not change fields that determine a key's equality or hash while that key is stored. The key can become inconsistent with its original bucket. Immutable string IDs avoid that mutation hazard. A custom business object may instead need a comparer based on an immutable ID, as introduced in Chapter 13.

A hash collection does not provide sorted order, range queries, or next-priority removal. For those requirements choose another structure, or maintain multiple indexes with a clear consistency policy. The next section coordinates dictionary membership with the priority queue from Chapter 18.


```{index} Practice: Report Missing IDs
```

## Practice: Report Missing IDs




```{index} Practice: Report Missing IDs
```


Produce planned minus actual without modifying the planned set.


In [ ]:
using System;
using System.Collections.Generic;

var planned=new HashSet<string>(new[]{"A","B","C"},StringComparer.Ordinal);
var actual=new HashSet<string>(new[]{"B","D"},StringComparer.Ordinal);
var missing=new HashSet<string>(planned,StringComparer.Ordinal);// Your code starts here.
/* TODO: keep only IDs absent from actual. */
// Your code ends here.
var report=new List<string>(missing);report.Sort(StringComparer.Ordinal);
Console.WriteLine(string.Join(",",report));


In [1]:
using System;
using System.Collections.Generic;

var planned=new HashSet<string>(new[]{"A","B","C"},StringComparer.Ordinal);
var actual=new HashSet<string>(new[]{"B","D"},StringComparer.Ordinal);
var missing=new HashSet<string>(planned,StringComparer.Ordinal);missing.ExceptWith(actual);
var report=new List<string>(missing);report.Sort(StringComparer.Ordinal);
Console.WriteLine(string.Join(",",report));


A,C


```{rubric} Footnotes
```


[^1902_dictionaries_sets-1]: Set difference is directional. Registered minus attended and attended minus registered answer different business questions.


[^development-reading]: Further reading: [OpenDSA, Introduction to Hashing](https://opendsax.cs.vt.edu/OpenDSA/Books/Everything/html/HashIntro.html). The worked business scenarios and guided reasoning here are original teaching examples.
